In [8]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import os
import glob
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [6]:
# Set font and plot styles in Matplotlib
plt.rcParams.update(
    {
        "font.family": "Roboto",
        # "font.weight": "medium",
        "legend.fontsize": 13,
        "legend.title_fontsize": 14,
        "font.size": 16,
        "axes.titlesize": 16,
        "axes.labelsize": 16,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "axes.edgecolor": "black",  # For black tick borders
        "axes.linewidth": 1.5,  # Make axis lines more visible
        # "axes.spines.left": True,  # Display the left spine
        # "axes.spines.bottom": True,  # Display the bottom spine
        # "axes.spines.right": True,  # Display the right spine
        # "axes.spines.top": True,  # Display the top spine
        # "lines.linewidth": 0.8,  # Set the default line width for plots
        "xtick.major.width": 1.5,  # X-axis major ticks
        "ytick.major.width": 1.5,  # Y-axis major ticks
        "lines.linewidth": 1.5,  # Lines in line plots
        "patch.linewidth": 1.5,  # For bar edge lines (used by histplot)
        "legend.edgecolor": "black",  # frame color for the legend
        "savefig.dpi": 300,  # Figure DPI while saving
    }
)

# Apply the "ticks" style manually by removing the grid
plt.style.use("seaborn-v0_8-deep")


# RACIPE:

In [12]:
results_dir = "SimulResults"

# Listing the results folders
#topo_result_dirs = sorted(glob.glob(results_dir + "/*/"))
topo_result_dirs = [results_dir + "/dorothea_trunc/"]#"/dorothea_tf2_abc/"]
print(topo_result_dirs)

# Loop through the results
for topo_result in topo_result_dirs:
    topo_name = os.path.basename(topo_result.rstrip("/")).replace("_BAK", "")
    print(topo_name)
    
    replicate_dirs = [
        path
        for path in glob.glob(topo_result + "/*/")
        if os.path.basename(path.rstrip("/")).isnumeric()
    ]
    # List of dataframe to concatenate at the end
    soldf = []
    paramdf = []
    steady_counts_df = []
    steady_type_df = []
    
    for rep_dir in replicate_dirs:
        rep_num = os.path.basename(rep_dir.rstrip("/"))
        print("Replicate Number :", rep_num)
        # Reading the solution dataframe
        sol_df = pd.read_parquet(
            os.path.join(
                rep_dir, f"{topo_name}_steadystate_solutions_{rep_num}.parquet"
            )
        )
        display(sol_df)

        param_df = pd.read_parquet(
            os.path.join(
                rep_dir, f"{topo_name}_params_{rep_num}.parquet"
            )
        )
        display(param_df)
        
        # Getting the gk columns
        gk_cols = [col for col in sol_df.columns if col.startswith("gk_")]
        gene_cols = [col.replace("gk_", "") for col in gk_cols]
        # Keep only rows where 'State' does NOT contain the substring 'nan'
        sol_df = sol_df[~sol_df["State"].str.contains("nan", na=False)]
        # Keep only the rows which reach steady state
        sol_df = sol_df[sol_df["SteadyStateFlag"] == 1]
        # Resetting the index to avoid merginf problem witht eh new redorder state column
        sol_df = sol_df.reset_index(drop=True)
        # Creating a state df
        nstate_df = sol_df["State"].str.strip("'").apply(list)
        nstate_df = pd.DataFrame(nstate_df.to_list(), columns=gene_cols)
        new_order = [
            "miR141",
            "miR101",
            "miR34a",
            "miR200a",
            "miR200c",
            "miR200b",
            "GSC",
            "TWIST1",
            "TWIST2",
            "FOXC2",
            "TGFbeta",
            "SNAI1",
            "SNAI2",
            "ZEB1",
            "ZEB2",
        ]
        display(nstate_df)
        nstate_df = nstate_df.astype(str).apply("".join, axis=1)
        sol_df["State"] = nstate_df
        print(sol_df)
        #     param_df = pd.read_parquet(
        #         os.path.join(rep_dir, f"{topo_name}_params_{rep_num}.parquet")
        #     )
        #     # sol_df = gk_normalise_solutions(sol_df, param_df, keep_gk=True)
        # Adding Rep number
        sol_df["RepNum"] = int(rep_num)
        soldf.append(sol_df)
        # reading the Steady state counts
        steady_counts = sol_df["State"].value_counts(normalize=True).reset_index()
        steady_counts.columns = ["State", "Fraction"]
        steady_counts["RepNum"] = int(rep_num)
        steady_counts_df.append(steady_counts)
        # Getting the unique number of steady states in for the parameter sets
        multi_stable_df = (
            sol_df.groupby("ParamNum")["State"]
            .nunique()
            .value_counts(normalize=True)
            .reset_index()
        )
        multi_stable_df.columns = ["Stability Type", "Fraction"]
        multi_stable_df["RepNum"] = int(rep_num)
        steady_type_df.append(multi_stable_df)
        
    
    soldf = pd.concat(soldf, axis=0)
    # gk_cols = [cl for cl in soldf.columns if "gk_" in cl]
    # print(soldf[gk_cols])
    # sns.histplot(
    #     sol_df[gk_cols].values.flatten(),
    #     bins=100,
    # )
    # plt.show()
    # plt.close()
    steady_counts_df = pd.concat(steady_counts_df, axis=0)
    steady_counts_df["Motif"] = topo_name
    # C print(steady_counts_df)
    steady_counts_df.to_csv(
        os.path.join(topo_result, f"{topo_name}_StateCounts.csv"), index=False
    )
    steady_type_df = pd.concat(steady_type_df, axis=0)
    steady_type_df["Motif"] = topo_name
    # print(steady_type_df)
    steady_type_df.to_csv(
        os.path.join(topo_result, f"{topo_name}_StateTypes.csv"), index=False
    )
    # Create the barplot
    plt.figure(figsize=(5, 6))
    steady_counts_df["State"] = steady_counts_df["State"].str.strip("'")
    steady_counts_df = steady_counts_df[steady_counts_df["Fraction"] >= 0.05]
    # Sort by Fraction
    steady_counts_df = steady_counts_df.sort_values("Fraction", ascending=False)
    ax = sns.barplot(
        data=steady_counts_df,
        x="State",
        y="Fraction",
        estimator="mean",  # Use mean across replicates
        errorbar="sd",  # For Seaborn >=0.12; use ci="sd" for older versions
        capsize=0.2,
        edgecolor="black",
        color=sns.color_palette("deep")[4],
    )
    plt.xticks(rotation=90)
    # Annotate each bar individually
    for bar in ax.patches:
        height = bar.get_height()
        x = bar.get_x() + bar.get_width() / 2
        label = f"{height:.2f}"

        # Adjust label position slightly above the bar (e.g., 5% of height)
        offset = height * 0.23
        ax.annotate(
            label,
            (x, height + offset),
            ha="center",
            va="bottom",
            fontsize=14,
        )
    ax.set_ylim(0, ax.get_ylim()[1] * 1.1)  # Increase top y-limit by 10%
    # plt.title(f"Steady State Distribution of {topo_name}")
    plt.ylabel("Fraction of States")
    plt.xlabel("State")
    plt.tight_layout()
    plt.savefig(os.path.join(topo_result, f"{topo_name}_StateCounts.png"), dpi=300)
    plt.savefig(os.path.join(topo_result, f"{topo_name}_StateCounts.svg"), dpi=300)
    plt.close()
    # plt.show()
    plt.figure(figsize=(5.5, 5))
    ax = sns.barplot(
        data=steady_type_df,
        x="Stability Type",
        y="Fraction",
        estimator="mean",  # Use mean across replicates
        errorbar="sd",  # For Seaborn >=0.12; use ci="sd" for older versions
        capsize=0.2,
        edgecolor="black",
        color=sns.color_palette("deep")[2],
    )
    for container in ax.containers:
        ax.bar_label(
            container,
            fmt="%.2f",  # <-- format to 2 decimal points
            padding=52,  # <-- adjust distance from top of bar
            fontsize=14,  # <-- optional: control font size
        )
    ax.set_ylim(0, ax.get_ylim()[1] * 1.1)  # Increase top y-limit by 10%
    # plt.title(f"Multi-Stability Type Distribution ofbb {topo_name}")
    plt.ylabel("Fraction of States")
    plt.xlabel("Number of Stable States")
    plt.tight_layout()
    # plt.savefig(os.path.join(topo_result, f"{topo_name}_StateType.svg"))
    plt.savefig(os.path.join(topo_result, f"{topo_name}_StateType.png"), dpi=300)
    plt.savefig(os.path.join(topo_result, f"{topo_name}_StateType.svg"), dpi=300)
    # plt.show()
    plt.close()
    # Running PCA
    #run_and_plot_pca(soldf, new_order, n_components=10)
    

['SimulResults/dorothea_trunc/']
dorothea_trunc
Replicate Number : 001


,ABCE1,ACAD10,ACKR2,ACP3,AFF1,AGTPBP1,AHR,AHSG,AKR1B1,AKR1C3,...,gk_ZCCHC2,gk_ZCCHC9,gk_ZFHX4,gk_ZNF131,gk_ZNF217,gk_ZNF385D,gk_ZNF559,State,ic_index,param_index
0,93.589500,45.643299,26.353800,14.835500,6.121200,13.487100,45.841499,19.972200,4.885500,60.629299,...,0.991263,0.013044,0.185452,0.178906,0.170048,0.135067,0.118405,'000000000000000000000000000000000000000000000...,0,0
1,0.280000,5.678000,73.326401,0.552900,85.038101,346.393005,384.376678,1.411300,0.558600,3.387300,...,0.985213,0.030360,0.010020,0.916224,0.999982,0.739432,0.010175,'000000000000000000000000000000000000000000000...,0,1
2,170.682297,19.802399,123.506393,35.995899,75.591896,63.120499,166.468994,42.332298,2.354100,2.852400,...,0.450609,0.976516,0.900122,0.962474,0.808204,0.467283,0.383549,'000000000000000000000000000000000000000000000...,0,2
3,88.783699,385.142578,19.265299,223.172195,16.789499,12.698700,266.633087,82.630096,429.348297,10.471900,...,0.094608,0.999923,0.999172,0.141422,0.701928,0.999365,0.884960,'000000000000000000000000000000000000000000000...,0,3
4,0.741500,6.175200,12.657000,47.382599,288.730804,18.396500,121.568398,8.601300,4.566400,94.519096,...,0.028454,0.382967,0.028810,0.728995,0.457629,0.994000,0.029319,'000000000000000000000000000000000000000000000...,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,1.359500,1.890800,61.380299,39.100399,244.501587,0.450000,149.819702,47.680000,35.408897,39.275200,...,0.045802,0.582130,0.557293,0.091371,0.752252,0.669517,0.580674,'000000000000000000000000000000000000000000000...,99,9995
999996,64.477402,47.370598,16.803200,84.399796,81.120201,12.712399,23.090099,28.578800,30.567999,11.785999,...,0.630192,0.988913,0.950343,0.749372,0.096119,0.011305,0.916564,'000000000000000000000000000000000000000000000...,99,9996
999997,95.472694,21.439999,59.757000,53.327198,60.307800,10.556700,183.592590,21.653599,270.475494,4.233800,...,0.097589,0.533770,0.736175,0.167119,0.629844,0.562181,0.055324,'000000000000000000000000000000000000000000000...,99,9997
999998,122.224098,124.743294,124.977997,40.358997,44.321499,105.530296,99.575897,103.135094,87.460999,6.797600,...,0.984127,0.999900,0.958497,0.682052,0.162546,0.295473,0.999614,'000000000000000000000000000000000000000000000...,99,9998


,Prod_ABCE1,Prod_ACAD10,Prod_ACKR2,Prod_ACP3,Prod_AFF1,Prod_AGTPBP1,Prod_AHR,Prod_AHSG,Prod_AKR1B1,Prod_AKR1C3,...,ActFld_AHR_ZNF217,Thr_AHR_ZNF217,Hill_AHR_ZNF217,ActFld_AHR_ZNF385D,Thr_AHR_ZNF385D,Hill_AHR_ZNF385D,ActFld_AR_ZNF559,Thr_AR_ZNF559,Hill_AR_ZNF559,ParamNum
0,95.388063,20.081085,19.838647,1.983397,6.792379,23.141167,43.247742,88.308135,77.738213,39.731116,...,34.257931,82.795422,3.0,54.577174,89.366264,3.0,9.574921,83.386172,3.0,1
1,12.486767,80.553548,53.878015,22.959993,72.057851,74.480113,58.044249,78.648243,14.693324,3.240163,...,33.529124,62.907115,6.0,96.864742,137.368270,1.0,98.313338,137.137040,3.0,2
2,96.770623,34.502295,88.551032,95.903535,99.702329,77.978020,96.093632,19.115815,3.200870,3.716375,...,69.930409,125.298972,5.0,47.396147,176.601358,3.0,54.705760,162.947209,2.0,3
3,32.939308,81.449893,4.236984,48.473034,12.713463,97.478353,98.490540,11.068968,84.567397,30.349231,...,38.733699,117.548478,1.0,28.977049,6.840852,2.0,30.229429,104.122803,3.0,4
4,39.210898,35.788728,74.096373,55.504510,74.198623,55.712765,73.517443,8.042457,31.978916,93.583997,...,14.721008,169.196456,1.0,32.031618,34.157060,4.0,34.257002,103.643159,6.0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,16.132081,9.929488,43.794189,99.039835,83.100946,7.198903,62.101682,78.352664,91.126109,64.153801,...,4.889909,122.864247,4.0,87.206264,130.540950,5.0,42.235706,81.028897,5.0,9996
9996,26.862604,14.614413,4.329224,61.356159,85.335810,37.633619,16.750879,20.373204,26.674617,21.933014,...,23.795816,59.031120,3.0,94.342228,77.181878,6.0,34.687428,153.599860,6.0,9997
9997,69.531728,20.232641,56.195861,43.785259,8.770417,68.426325,75.978347,68.810371,98.008640,61.490405,...,91.217549,141.985218,2.0,98.410383,145.582002,1.0,97.619625,161.219572,5.0,9998
9998,71.062482,22.753457,57.181815,43.783538,46.455032,94.615323,86.856414,93.146603,68.768205,46.863249,...,65.817687,153.746430,4.0,38.456977,137.178449,3.0,72.424087,31.747561,4.0,9999


,ABCE1,ACAD10,ACKR2,ACP3,AFF1,AGTPBP1,AHR,AHSG,AKR1B1,AKR1C3,...,ZBTB16,ZBTB7B,ZC3H3,ZCCHC2,ZCCHC9,ZFHX4,ZNF131,ZNF217,ZNF385D,ZNF559
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
999996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
999997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
999998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


             ABCE1      ACAD10       ACKR2        ACP3        AFF1  \
0        93.589500   45.643299   26.353800   14.835500    6.121200   
1         0.280000    5.678000   73.326401    0.552900   85.038101   
2       170.682297   19.802399  123.506393   35.995899   75.591896   
3        88.783699  385.142578   19.265299  223.172195   16.789499   
4         0.741500    6.175200   12.657000   47.382599  288.730804   
...            ...         ...         ...         ...         ...   
999995    1.359500    1.890800   61.380299   39.100399  244.501587   
999996   64.477402   47.370598   16.803200   84.399796   81.120201   
999997   95.472694   21.439999   59.757000   53.327198   60.307800   
999998  122.224098  124.743294  124.977997   40.358997   44.321499   
999999  110.029396  116.827400  340.315582  157.521393    1.230900   

           AGTPBP1         AHR        AHSG      AKR1B1     AKR1C3  ...  \
0        13.487100   45.841499   19.972200    4.885500  60.629299  ...   
1       346

/tmp/ipykernel_363635/3559090764.py:156: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


In [14]:
print(list(sol_df.columns))

['AHR', 'AR', 'ARID2', 'ARID3A', 'ARNT', 'ARNTL', 'ASCL1', 'ATF1', 'ATF2', 'ATF3', 'ATF4', 'ATF6', 'ATF7', 'BACH1', 'BACH2', 'BATF', 'BCL11A', 'BCL6', 'BHLHE40', 'CDX2', 'CEBPA', 'CEBPB', 'CEBPD', 'CEBPG', 'CLOCK', 'CREB1', 'CREB3', 'CREB3L1', 'CREM', 'CTCF', 'CTCFL', 'CUX1', 'DUX4', 'E2F1', 'E2F2', 'E2F3', 'E2F4', 'E2F5', 'E2F6', 'E2F7', 'EBF1', 'EGR1', 'EHF', 'ELF1', 'ELF3', 'ELF5', 'ELK1', 'ELK4', 'EPAS1', 'ERG', 'ESR1', 'ESR2', 'ESRRA', 'ETS1', 'ETS2', 'ETV1', 'ETV4', 'FLI1', 'FOS', 'FOSL1', 'FOSL2', 'FOXA1', 'FOXA2', 'FOXJ2', 'FOXK2', 'FOXL2', 'FOXM1', 'FOXO1', 'FOXO3', 'FOXO4', 'FOXP1', 'FOXP2', 'GABPA', 'GATA1', 'GATA2', 'GATA3', 'GATA4', 'GATA6', 'GFI1B', 'GLI2', 'GRHL2', 'HBP1', 'HHEX', 'HIF1A', 'HINFP', 'HMBOX1', 'HNF1A', 'HNF1B', 'HNF4A', 'HNF4G', 'HOXA9', 'HOXB13', 'HSF1', 'IKZF1', 'IRF1', 'IRF2', 'IRF3', 'IRF4', 'IRF9', 'JUN', 'JUNB', 'JUND', 'KDM5B', 'KLF1', 'KLF13', 'KLF4', 'KLF5', 'KLF6', 'KLF9', 'KMT2A', 'LEF1', 'LHX2', 'LYL1', 'MAF', 'MAFB', 'MAFF', 'MAFG', 'MAFK', 'M

In [17]:
param_df = pd.read_parquet(
    os.path.join(
        rep_dir, f"{topo_name}_params_{rep_num}.parquet"
    )
)

ic_df = pd.read_parquet(
    os.path.join(
        rep_dir, f"{topo_name}_init_conds_{rep_num}.parquet"
    )
)

In [19]:
display(sol_df)

,AHR,AR,ARID2,ARID3A,ARNT,ARNTL,ASCL1,ATF1,ATF2,ATF3,...,gk_ZFX,gk_ZNF143,gk_ZNF217,gk_ZNF263,gk_ZNF274,gk_ZNF384,gk_ZNF639,gk_ZNF740,State,RepNum
0,1.8439,0.664600,15.016500,0.9370,2.2257,117.903694,416.227997,4.4805,0.9510,0.0,...,1.000000,0.140261,0.0,0.020200,0.999998,1.000000,1.000001,1.000000,0010011000000001000000111011011010001100000100...,1
1,0.2101,4.574800,152.577499,0.1110,21.2267,2.609800,137.898300,0.2547,0.2887,0.0,...,1.000000,0.095940,0.0,0.012780,1.000000,1.000002,1.000001,1.000000,0010001000010001000000111011001010001100000100...,1
2,0.0209,10.792399,10.408199,7.8314,0.9479,21.896000,66.721100,0.7940,0.4743,0.0,...,1.000000,0.047376,0.0,0.016040,1.000000,1.000000,0.999999,1.000002,0010011000010001000000111011001010001100000000...,1
3,0.0721,0.503000,91.066200,3.6539,1.2671,1.353500,0.454300,11.0348,2.2921,0.0,...,1.000000,0.293133,0.0,0.030640,1.000000,0.999999,1.000000,1.000000,0010000000010001000000111011001010001000000000...,1
4,1.1221,0.583700,137.816391,0.0128,1.6074,36.310898,42.577599,5.2985,0.4422,0.0,...,1.000000,0.429884,0.0,0.004239,1.000000,1.000000,1.000000,1.000000,0010011000000001000000111011001010001000000100...,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
858375,0.0049,2.861600,105.712601,0.0956,2.9330,4.907600,218.289200,2.0275,7.9826,0.0,...,1.000000,0.131147,0.0,0.000224,1.000000,1.000000,0.999999,1.000000,0010001000010001000000111011001010001100000100...,1
858376,0.1138,0.007900,303.198395,0.0117,0.1127,4.113900,41.473297,3.1705,0.6682,0.0,...,1.000000,0.508955,0.0,0.011397,0.999999,1.000000,1.000000,1.000004,0010001000000001000000111011001010001100000000...,1
858377,0.0440,0.055500,123.607796,0.4097,0.4908,4.406800,31.737999,1.7073,4.2862,0.0,...,1.000000,0.991296,0.0,0.003884,1.000001,1.000000,0.999988,1.000000,0010001000000001000000111011001010001000000100...,1
858378,0.1191,0.256500,218.446198,0.0055,0.9926,1.285800,132.067200,1.1003,7.3366,0.0,...,1.000000,0.045158,0.0,0.018170,0.999999,1.000000,0.999999,1.000000,0010001000000001000000111011001010001000000000...,1


In [18]:
display(param_df)
display(ic_df)

,Prod_AHR,Prod_AR,Prod_ARID2,Prod_ARID3A,Prod_ARNT,Prod_ARNTL,Prod_ASCL1,Prod_ATF1,Prod_ATF2,Prod_ATF3,...,InhFld_MBD1_ZNF263,Thr_MBD1_ZNF263,Hill_MBD1_ZNF263,ActFld_NRF1_ZNF263,Thr_NRF1_ZNF263,Hill_NRF1_ZNF263,InhFld_ZNF274_ZNF263,Thr_ZNF274_ZNF263,Hill_ZNF274_ZNF263,ParamNum
0,95.666774,586263.996045,14.246834,70.513230,39.739279,73.964796,66.964781,76.173698,43.271446,51930.333516,...,0.034842,27.041084,4.0,57.261227,44.412525,2.0,0.010798,142.329195,4.0,1
1,99.050479,898988.494254,61.490838,51.472832,79.422165,76.239792,83.634717,20.320600,27.770148,23747.868049,...,0.046631,7.428544,6.0,74.347580,71.386340,2.0,0.014272,97.275261,2.0,2
2,76.266487,668587.888022,9.551861,99.231349,19.584231,20.601501,67.309616,14.376048,9.910074,10424.009484,...,0.039854,11.860897,4.0,61.900742,64.849626,5.0,0.033275,174.888168,2.0,3
3,91.422499,744565.975846,67.874153,86.949178,25.690666,93.030118,20.187930,74.187067,77.547398,70928.690392,...,0.011589,39.283544,2.0,18.787443,65.902357,2.0,0.100030,163.949965,2.0,4
4,93.997932,399613.162667,96.948190,35.497035,52.805535,8.143863,38.079365,80.532997,15.013193,21223.853574,...,0.048688,65.021208,4.0,53.123086,65.926518,2.0,0.021662,96.238677,2.0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,53.655635,592378.147202,22.594069,20.505471,24.697079,8.822740,35.625467,53.333929,84.276021,48007.561108,...,0.023700,57.730966,4.0,19.708047,58.420288,3.0,0.014719,13.917501,2.0,9996
9996,73.825826,649142.052429,25.989189,8.135876,55.173151,18.146231,95.927213,68.162196,60.859552,14935.671065,...,0.020630,38.472331,4.0,44.975650,47.511805,3.0,0.032422,104.813463,3.0,9997
9997,77.294810,473373.243710,67.084102,56.482113,74.107900,98.157738,30.504320,33.472288,8.560520,68058.017054,...,0.011569,8.373118,2.0,67.031303,60.460098,4.0,0.011087,143.297899,5.0,9998
9998,15.877177,226100.053558,15.728586,14.640515,89.584696,21.810081,96.639323,91.767575,85.206694,4257.557278,...,0.010589,27.764897,4.0,91.101044,61.331663,3.0,0.011754,26.540377,1.0,9999


,AHR,AR,ARID2,ARID3A,ARNT,ARNTL,ASCL1,ATF1,ATF2,ATF3,...,ZEB2,ZFX,ZNF143,ZNF217,ZNF263,ZNF274,ZNF384,ZNF639,ZNF740,InitCondNum
0,56.965404,14.732645,20.835454,25.030464,28.655560,15.712576,9.974981,29.324461,69.828886,12.615664,...,10.708674,34.655890,16.679075,65.671284,21.085169,2.770465,88.315433,75.073585,93.810922,1
1,49.045367,74.612498,70.782367,86.581415,81.433381,94.503483,83.028203,58.616247,6.702159,78.782055,...,87.256079,56.144946,78.139661,47.381200,83.375123,93.235925,31.622889,8.563701,10.963696,2
2,22.470020,48.324323,34.367756,26.529522,63.762719,75.237132,48.087360,78.072364,47.609639,67.711732,...,47.920068,98.257579,48.075603,11.852794,54.761872,65.798000,53.939254,39.440589,33.568680,3
3,85.713120,90.532905,81.994782,63.422616,17.323081,41.299547,73.284138,11.270205,78.251982,48.824168,...,74.967662,14.489645,59.117755,80.186960,43.502942,31.007937,15.698341,81.200092,63.665017,4
4,97.289291,32.669219,52.697566,51.468761,4.619101,11.086175,34.287035,92.621387,36.987744,94.911480,...,21.331446,47.047825,95.946427,29.535958,28.065592,46.445807,11.377645,35.183004,22.178976,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,62.666332,78.186518,11.146172,9.215908,22.384315,85.113223,56.107926,63.330730,11.093913,80.905191,...,5.119544,19.938530,38.219346,85.272736,29.867135,53.658598,58.589084,62.898454,58.082977,96
96,60.858044,5.292386,38.334520,36.787450,80.347827,88.151722,33.118902,55.368472,77.315085,4.819601,...,25.390545,56.997028,89.020369,6.902967,47.185772,51.148700,1.827350,48.240898,26.664589,97
97,46.732840,59.301248,90.601853,75.207822,27.805214,22.122499,56.745390,32.451338,48.208251,87.158291,...,97.297820,29.356916,5.798355,87.467675,57.272673,42.661768,67.813258,83.825358,70.754814,98
98,24.969948,33.007762,3.140350,13.636668,16.816773,47.649054,24.967030,2.118932,5.705570,73.474996,...,37.878563,24.477956,70.095143,70.767831,81.251224,14.425786,42.400117,65.493413,97.516978,99


In [69]:
format(255,"03b")[:3]

'111'

# Boolise:

In [3]:
result_dir = "SimulResultsB"

# Specifying the topo name
topo_name = "dorothea_tf2_abc"

In [4]:
res_ic_list = [] # A dataframe mapping an index to each initial condition vector
res_steady_list = [] # A dataframe of all steady states depending on the inital conditions
res_counts_list = [] # A dataframe of the fraction of ICs for each unique steady state

for updatever in ["sync","async"]:
    for replicate in range(1,4):#4):
        ising_df = pd.read_parquet(
            f"./{result_dir}/{topo_name}/{replicate}/{topo_name}_{updatever}_ising_results.parquet"
        )
        # Get the max step value, as things start from 0 it will be the max value of step column + 1
        print(ising_df["Step"].max() + 1)
        # Since 8 genes are combined in each column, their numeric representation
        for eight_genes in ising_df.columns[2:]:
            ising_df[eight_genes] = [format(int(ic),"08b")[:eight_genes.count("|")+1] for ic in list(ising_df[eight_genes])] # Turn numeric representations into 8-length bools
        ising_df["State"] = ising_df[ising_df.columns[2:]].astype(str).agg("".join, axis=1)
        steady_df = ising_df[["Initnum","Step","State"]]
        steady_df["Replicate"] = replicate
        steady_df["Mode"] = updatever.capitalize()

        ic_df = steady_df[steady_df["Step"]==0]
        steady_df = steady_df[steady_df["Step"]!=0]
        res_ic_list.append(ic_df.drop("Step",axis=1))
        res_steady_list.append(steady_df.drop("Step",axis=1))

        state_counts = ising_df["State"].value_counts(normalize=True).reset_index()
        state_counts.columns = ["State", "Fraction"]
        state_counts["Replicate"] = replicate
        state_counts["Mode"] = updatever.capitalize()
        res_counts_list.append(state_counts)

res_ic_df = pd.concat(res_ic_list)
res_ic_df.to_csv(
    f"./{result_dir}/{topo_name}/{topo_name}_SteadyStates_ICs.csv", index=False
)

res_steady_df = pd.concat(res_steady_list)
res_steady_df.to_csv(
    f"./{result_dir}/{topo_name}/{topo_name}_SteadyStates.csv", index=False
)


res_counts_df = pd.concat(res_counts_list)
res_counts_df.to_csv(
    f"./{result_dir}/{topo_name}/{topo_name}_StateCounts_Main.csv", index=False
)

2591


/tmp/ipykernel_363635/1415968792.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  steady_df["Replicate"] = replicate
/tmp/ipykernel_363635/1415968792.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  steady_df["Mode"] = updatever.capitalize()


2591


/tmp/ipykernel_363635/1415968792.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  steady_df["Replicate"] = replicate
/tmp/ipykernel_363635/1415968792.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  steady_df["Mode"] = updatever.capitalize()


2591


...

/tmp/ipykernel_363635/1415968792.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  steady_df["Replicate"] = replicate
/tmp/ipykernel_363635/1415968792.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  steady_df["Mode"] = updatever.capitalize()


2591


/tmp/ipykernel_363635/1415968792.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  steady_df["Replicate"] = replicate
/tmp/ipykernel_363635/1415968792.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  steady_df["Mode"] = updatever.capitalize()


2591


/tmp/ipykernel_363635/1415968792.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  steady_df["Replicate"] = replicate
/tmp/ipykernel_363635/1415968792.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  steady_df["Mode"] = updatever.capitalize()
